# Training a Small Music Transformer from Scratch

In this notebook we:
1. Tokenize MIDI files using MidiTok (REMI tokenization)
2. Build a small GPT-style transformer (fits on T4 GPU)
3. Train it on the tokenized data
4. Generate new music and decode back to MIDI

This gives you hands-on understanding of how Music Transformer-style models work.

In [ ]:
!pip install miditok torch matplotlib pretty_midi numpy IPython

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pretty_midi
from pathlib import Path

## Step 1: Create Training Data

We'll create a small dataset of MIDI melodies programmatically (in a real project, you'd use an existing MIDI dataset like the Lakh MIDI Dataset).

In [ ]:
def create_melody(seed, length=64, scale=[60,62,64,65,67,69,71,72]):
    np.random.seed(seed)
    midi = pretty_midi.PrettyMIDI(initial_tempo=120)
    piano = pretty_midi.Instrument(program=0)
    t = 0
    prev = np.random.choice(scale)
    for _ in range(length):
        weights = np.array([1.0/(abs(n-prev)+1) for n in scale])
        weights /= weights.sum()
        pitch = np.random.choice(scale, p=weights)
        dur = np.random.choice([0.25, 0.5, 0.5, 1.0])
        piano.notes.append(pretty_midi.Note(80, pitch, t, t+dur*0.9))
        prev = pitch
        t += dur
    midi.instruments.append(piano)
    return midi

# Create 20 training melodies
Path('train_midi').mkdir(exist_ok=True)
for i in range(20):
    m = create_melody(seed=i, length=128)
    m.write(f'train_midi/melody_{i:02d}.mid')
print(f"Created 20 training MIDI files")

## Step 2: Tokenize with MidiTok

In [ ]:
from miditok import REMI, TokenizerConfig

config = TokenizerConfig(
    num_velocities=16,
    use_chords=False,
    use_programs=False,
)
tokenizer = REMI(config)

# Tokenize all files
tokens_list = []
for f in sorted(Path('train_midi').glob('*.mid')):
    tokens = tokenizer(f)
    tokens_list.append(tokens.ids[0])  # First track

# Show vocabulary
print(f"Vocabulary size: {len(tokenizer)}")
print(f"Total tokens: {sum(len(t) for t in tokens_list)}")
print(f"Average sequence length: {np.mean([len(t) for t in tokens_list]):.0f}")
print(f"\nSample tokens (first 20): {tokens_list[0][:20]}")

## Step 3: Build Dataset and Model

In [ ]:
class MusicDataset(Dataset):
    def __init__(self, token_sequences, seq_len=128):
        self.seq_len = seq_len
        self.data = []
        for seq in token_sequences:
            for i in range(0, len(seq) - seq_len - 1, seq_len // 2):
                self.data.append(torch.tensor(seq[i:i+seq_len+1], dtype=torch.long))
    
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        seq = self.data[idx]
        return seq[:-1], seq[1:]  # input, target

class MiniMusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=4, seq_len=128):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)
        
        # Causal mask
        self.register_buffer('mask', torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1))
    
    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        x = self.embed(x) + self.pos_embed(pos)
        x = self.transformer(x, mask=self.mask[:T, :T], is_causal=True)
        return self.head(x)
    
    def generate(self, prompt, max_new=256, temperature=1.0, top_k=50):
        self.eval()
        tokens = prompt.clone()
        for _ in range(max_new):
            x = tokens[:, -128:]  # Window
            logits = self(x)[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            tokens = torch.cat([tokens, next_token], dim=1)
        return tokens

# Create dataset and model
dataset = MusicDataset(tokens_list, seq_len=128)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

vocab_size = len(tokenizer)
model = MiniMusicTransformer(vocab_size, d_model=128, nhead=4, num_layers=4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} parameters")
print(f"Dataset: {len(dataset)} sequences")
print(f"Device: {device}")

## Step 4: Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
losses = []

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), batch_y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

## Step 5: Generate

In [ ]:
# Use first few tokens as prompt
prompt = torch.tensor([tokens_list[0][:16]], dtype=torch.long).to(device)

with torch.no_grad():
    generated = model.generate(prompt, max_new=256, temperature=0.9, top_k=50)

gen_tokens = generated[0].cpu().tolist()
print(f"Generated {len(gen_tokens)} tokens")

# Decode back to MIDI
from miditok import TokSequence
tok_seq = TokSequence(ids=gen_tokens)
midi_out = tokenizer.decode(tok_seq)
midi_out.dump_midi('generated_transformer.mid')
print("Saved generated_transformer.mid")

# Visualize
midi = pretty_midi.PrettyMIDI('generated_transformer.mid')
pr = midi.get_piano_roll(fs=20)
plt.figure(figsize=(15, 4))
plt.imshow(pr[48:84], aspect='auto', origin='lower', cmap='magma')
plt.xlabel('Time'); plt.ylabel('Note')
plt.title('Transformer-Generated Music')
plt.colorbar()
plt.show()

## Step 6: Compare

Compare statistics of generated vs training data.

In [ ]:
# Load training and generated MIDI
train_midi = pretty_midi.PrettyMIDI('train_midi/melody_00.mid')
gen_midi = pretty_midi.PrettyMIDI('generated_transformer.mid')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, midi, label in [(axes[0], train_midi, 'Training'), (axes[1], gen_midi, 'Generated')]:
    notes = [n.pitch for inst in midi.instruments for n in inst.notes]
    ax.hist(notes, bins=range(48, 85), alpha=0.7, color='steelblue')
    ax.set_xlabel('MIDI Note')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} — Pitch Distribution')

plt.tight_layout()
plt.show()